# Mass Spectrometry Quality Control

This notebook demonstrates how to use the `ms_qc_tools` package to calculate and interpret various quality control metrics for mass spectrometry data.

In [ ]:
from ms_qc_tools.core import MSQualityControl
import pandas as pd
import json

def print_nice(data):
    """Helper function to nicely print dictionaries or other data types."""
    if isinstance(data, dict):
        print(json.dumps(data, indent=4, default=str))
    else:
        print(data)

qc = MSQualityControl(mzxml_filename=mzxml_file, psm_filename=psm_file)

## 1. Peptide Metrics

**Metrics:**
- `unique_peptide_count`, `unique_protein_count`: Number of unique identifications.
- `avg_peptide_length`, etc.: Peptide length statistics.
- `missed_cleavages_count`, `avg_missed_cleavages`: Missed cleavages statistics. High missed cleavages may indicate digestion issues.
- `avg_hydrophobicity`: Average peptide hydrophobicity.
- `modified_peptide_pct`: Percentage of modified peptides.
- `charge_distribution`: Distribution of peptide charge states.
- `peptide_confidence_metrics`: Statistics on peptide confidence scores (e.g., mean, median, distribution).

In [ ]:
print("Peptide Property Metrics:")
print_nice(qc.metrics.peptide())
print("\nPeptide Charge Distribution:")
print_nice(qc.metrics.peptide_charge_distribution())
print("\nPeptide Confidence Metrics:")
print_nice(qc.metrics.peptide_confidence_metrics())

In [ ]:
_ = qc.plots.peptide_length_distribution()

In [ ]:
_ = qc.plots.amino_acid_composition()

In [ ]:
_ = qc.plots.missed_cleavages()

In [ ]:
_ = qc.plots.semitryptic_peptides()

In [ ]:
_ = qc.plots.hydrophobicity_distribution()

In [ ]:
_ = qc.plots.modification_summary()

In [ ]:
_ = qc.plots.peptide_charge_distribution()

## 2. Identification Metrics

**Metrics:**
- `overall_id_rate`: Percentage of MS2 scans identified.
- `id_rate_by_precursor_intensity`: ID rate binned by precursor intensity. Helps identify sensitivity limits.
- `id_rate_by_charge`: ID rate binned by charge state.
- `calibration`: Mean mass error (ppm) per 100 m/z bin. Should be close to 0 and stable across the m/z range.

In [ ]:
print("ID Rate by Metric:")
print_nice(qc.metrics.id_rate_by_metric(ms_level=2))
print("\nCalibration Deltas:")
print_nice(qc.metrics.calibration())

In [ ]:
_ = qc.plots.id_rate_by_metric("precursor_intensity")

In [ ]:
_ = qc.plots.id_rate_by_metric("fragment_count")

In [ ]:
_ = qc.plots.id_rate_by_metric("charge")

In [ ]:
_ = qc.plots.id_rate_by_metric("rt")

In [ ]:
_ = qc.plots.calibration_deltas()

In [ ]:
_ = qc.plots.ms1_feature_map_with_psms()

## Load into database

In [ ]:
if "_QC_" in str(raw_file):
    qc.save_metrics_to_db(db_path=metrics_db,
                          raw_filename=raw_file,
                         instrument=False,
                         identification=True)
else:
    print(f"Skipping database save: '{raw_file}' does not contain '_QC_'.")